## STAGE 1: DATA LOADING

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Loading reference data
subjects = pd.read_csv('raw_data/Идентификаторы Субъектов 10.04.2025.csv', sep=';')
mo = pd.read_csv('raw_data/Идентификаторы МО 15.04.2025.csv')

# Loading patient-program links
patient_program = pd.read_csv('raw_data/МО и Пациенты 10.04.2025.csv', sep=';')
# Clean patient ID from all types of whitespace
patient_program['id пациента'] = (patient_program['id пациента'].astype(str).str.replace(r'\s+', '', regex=True))  # all whitespace characters.str.replace('\xa0', '', regex=False)  # non-breaking space.astype(int))
# Loading primary data (with chunks for large files)
primary_chunks = []
chunk_size = 500000
for chunk in pd.read_csv('raw_data/Первичные данные от 10.04.2025.csv', chunksize=chunk_size, sep=','):
    primary_chunks.append(chunk)
primary = pd.concat(primary_chunks, ignore_index=True)
therapy = pd.read_csv('raw_data/Медикаментозная терапия от 10.04.2025.csv')
kzs = pd.read_csv('raw_data/Клинически значимые события от 15.04.2025.csv')
# Clear memory from chunks
primary_chunks.clear()
del primary_chunks
print(f"\nLoaded:")
print(f"  Subjects: {len(subjects)} rows")
print(f"  MO: {len(mo)} rows")
print(f"  Patient-program: {len(patient_program)} rows")
print(f"  Primary data: {len(primary):,} rows")
print(f"  Therapy: {len(therapy):,} rows")
print(f"  CSE: {len(kzs):,} rows")

## STAGE 2: DATA INTEGRITY CHECK

In [ ]:
# Add subject id to patient_program by name
print("Real columns in subjects:", subjects.columns.tolist())
print("Real columns in patient_program:", patient_program.columns.tolist())
patient_program['id пациента'] = patient_program['id пациента'].astype(str).str.replace(' ', '').astype(int)
subject_dict = subjects.set_index('название субъекта')['id субъекта'].to_dict()
# Add subject id via map (faster and more memory efficient)
patient_program['id субъекта'] = patient_program['название субъекта'].map(subject_dict)
mo_in_patient = patient_program['id МО'].isin(mo['id МО']).mean()
print(f"  % of MO ids from patient_program present in MO reference: {mo_in_patient*100:.1f}%")
subj_in_patient = patient_program['id субъекта'].notna().mean()
print(f"  % of records with found subject id: {subj_in_patient*100:.1f}%")
print(f"Duplicates in subjects: {subjects.duplicated().sum()}")
print(f"Duplicates in mo: {mo.duplicated().sum()}")
print(f"Duplicates in patient_program: {patient_program.duplicated().sum()}")
print(f"Duplicates in primary (patient id + measurement time): {primary.duplicated(['id пациента', 'время измерения']).sum():,}")
import gc
gc.collect()

## STAGE 3: DATA CLEANING

In [ ]:
# 3.1 Handling missing values
print("\n1. Missing values in primary data:")
primary_null = primary.isnull().sum()
primary_null_pct = (primary_null / len(primary)) * 100
null_df = pd.DataFrame({'Missing': primary_null, '%': primary_null_pct})
print(null_df[null_df['Missing'] > 0])
del primary_null, primary_null_pct, null_df
# 3.2 Data formats
# Date/Time
primary['время измерения'] = pd.to_datetime(primary['время измерения'], errors='coerce')
primary['время сохранения на сервере'] = pd.to_datetime(primary['время сохранения на сервере'], errors='coerce')
primary['дата рождения пациента'] = pd.to_datetime(primary['дата рождения пациента'], errors='coerce')
therapy['дата начала программы'] = pd.to_datetime(therapy['дата начала программы'], errors='coerce')
therapy['дата назначения'] = pd.to_datetime(therapy['дата назначения'], errors='coerce')
therapy['дата начала приема'] = pd.to_datetime(therapy['дата начала приема'], errors='coerce')
therapy['дата окончания приема'] = pd.to_datetime(therapy['дата окончания приема'], errors='coerce')
kzs['дата, время формирования КЗС'] = pd.to_datetime(kzs['дата, время формирования КЗС'], errors='coerce')

# Numeric fields
primary['САД'] = pd.to_numeric(primary['САД'], errors='coerce')
primary['ДАД'] = pd.to_numeric(primary['ДАД'], errors='coerce')
primary['ЧП'] = pd.to_numeric(primary['ЧП'], errors='coerce')
primary['рост'] = pd.to_numeric(primary['рост'], errors='coerce')
primary['масса'] = pd.to_numeric(primary['масса'], errors='coerce')

# Calculate age at measurement time
primary['возраст'] = (primary['время измерения'] - primary['дата рождения пациента']).dt.days / 365.25
import gc
gc.collect()

## STAGE 4: OUTLIER ANALYSIS

In [ ]:
# 4.1 Physiological boundaries
print("\n1. Checking physiological boundaries:")

# SBP
invalid_sbp = (primary['САД'] < 40) | (primary['САД'] > 250)
print(f"  SBP outside norm (40-250): {invalid_sbp.sum():,} ({invalid_sbp.mean()*100:.5f}%)")

# DBP
invalid_dbp = (primary['ДАД'] < 30) | (primary['ДАД'] > 150) | (primary['ДАД'] > primary['САД'])
print(f"  DBP outside norm (30-150) or >SBP: {(invalid_dbp).sum():,} ({invalid_dbp.mean()*100:.5f}%)")

# HR
invalid_hr = (primary['ЧП'] < 30) | (primary['ЧП'] > 220)
print(f"  HR outside norm (30-220): {invalid_hr.sum():,} ({invalid_hr.mean()*100:.5f}%)")

# Age
invalid_age = (primary['возраст'] < 18) | (primary['возраст'] > 120)
print(f"  Age outside norm (18-120): {invalid_age.sum():,} ({invalid_age.mean()*100:.5f}%)")

# 4.2 Temporal anomalies
print("\n2. Checking temporal anomalies:")

# Future measurements
future_meas = primary['время измерения'] > datetime.now()
print(f"  Future measurements: {future_meas.sum():,} ({future_meas.mean()*100:.5f}%)")

# Too frequent measurements (per patient per day)
primary['date'] = primary['время измерения'].dt.date
patient_daily = primary.groupby(['id пациента', 'date']).size().reset_index(name='n_meas')
freq_meas = patient_daily[patient_daily['n_meas'] > 20]
print(f"  Patient-days with >20 measurements: {len(freq_meas)} ({len(freq_meas)/len(patient_daily)*100:.5f}%)")

primary.drop('date', axis=1, inplace=True) 
del patient_daily
del freq_meas
import gc
gc.collect()

## STAGE 5: ANALYSIS BY GROUPS

In [ ]:
unique_patients = patient_program[['id пациента', 'группа наблюдения']].drop_duplicates('id пациента')
primary = primary.merge(
    unique_patients,
    on='id пациента',
    how='left'
)
del unique_patients
print("\n1. Group distribution:")
group_dist = primary['группа наблюдения'].value_counts(dropna=False)
print(group_dist)
print(f"\nTotal patients in groups: {group_dist.sum():,}")
print("\n2. Measurement statistics by group:")
group_stats_nunique = primary.groupby('группа наблюдения')['id пациента'].nunique()
group_stats_sad = primary.groupby('группа наблюдения')['САД'].agg(['mean', 'std', 'count']).round(1)
group_stats_dad = primary.groupby('группа наблюдения')['ДАД'].agg(['mean', 'std']).round(1)
group_stats_chp = primary.groupby('группа наблюдения')['ЧП'].agg(['mean', 'std']).round(1)
group_stats = pd.concat([
    group_stats_nunique.rename('count_patients'),
    group_stats_sad,
    group_stats_dad,
    group_stats_chp
], axis=1)
del group_stats_nunique, group_stats_sad, group_stats_dad, group_stats_chp
print(group_stats)

## STAGE 6: VISUALIZATION

In [ ]:
# Create fixed samples once
sample_size = min(100000, len(primary))
sample_indices = np.random.RandomState(42).choice(len(primary), sample_size, replace=False)
primary_sample = primary.iloc[sample_indices]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. SBP distribution
axes[0, 0].hist(primary['САД'].dropna(), bins=50, alpha=0.7, edgecolor='black')
axes[0, 0].set_title('SBP Distribution')
axes[0, 0].set_xlabel('SBP, mm Hg')
axes[0, 0].set_ylabel('Frequency')

# 2. DBP distribution
axes[0, 1].hist(primary['ДАД'].dropna(), bins=50, alpha=0.7, edgecolor='black', color='green')
axes[0, 1].set_title('DBP Distribution')
axes[0, 1].set_xlabel('DBP, mm Hg')
axes[0, 1].set_ylabel('Frequency')

# 3. SBP vs DBP relationship (using fixed sample)
axes[1, 0].scatter(primary_sample['САД'], primary_sample['ДАД'], alpha=0.1, s=1)
axes[1, 0].set_title(f'SBP vs DBP Relationship (sample {sample_size})')
axes[1, 0].set_xlabel('SBP, mm Hg')
axes[1, 0].set_ylabel('DBP, mm Hg')

# 4. Measurements by time of day (using same sample)
axes[1, 1].hist(primary_sample['время измерения'].dt.hour.dropna(), bins=24, alpha=0.7, color='orange')
axes[1, 1].set_title('Distribution of Measurements by Hour of Day')
axes[1, 1].set_xlabel('Hour of Day')
axes[1, 1].set_ylabel('Number of Measurements')

plt.tight_layout()
plt.savefig('data/photo/eda_basic_plots.png', dpi=100) # Clear memory
plt.close(fig)
del primary_sample
del sample_indices

## STAGE 7: CSE AND THERAPY ANALYSIS

In [ ]:
# CSE analysis
print("\n1. Top-10 CSE codes:")
kzs_counts = kzs['код КЗС'].value_counts().head(10)
print(kzs_counts)
del kzs_counts

kzs_per_patient = kzs.groupby('id пациента').size()
print(f"\n2. CSE statistics per patient:")
print(f"  Mean: {kzs_per_patient.mean():.1f}")
print(f"  Median: {kzs_per_patient.median():.1f}")
print(f"  Max: {kzs_per_patient.max():,}")
print(f"  % of patients with CSE: {(len(kzs_per_patient) / patient_program['id пациента'].nunique() * 100):.1f}%")
del kzs_per_patient
# Therapy analysis
print(f"\n3. Total therapy records: {len(therapy):,}")
if 'МНН' in therapy.columns:
    print("\n4. Top-10 prescribed INNs:")
    mnn_counts = therapy['МНН'].value_counts().head(10)
    print(mnn_counts)
    del mnn_counts
import gc
gc.collect()

## STAGE 8: SAVE AND STATISTICS

In [ ]:
clean_primary = primary[~invalid_sbp & ~invalid_dbp & ~invalid_hr & ~invalid_age & ~future_meas].copy()
print(f"Outliers removed: {len(primary) - len(clean_primary)} ({100 - len(clean_primary)/len(primary)*100:.1f}%)")
del invalid_sbp  
del invalid_dbp
clean_primary.to_csv('data/primary_clean.csv', index=False)
patient_program.to_csv('data/patient_program.csv', index=False)
kzs.to_csv('data/kzs.csv', index=False)
therapy.to_csv('data/therapy.csv', index=False)

print(f"""
Statistics:
- Patients: {patient_program['id пациента'].nunique():,}
- Measurements: {len(clean_primary):,}
- CSEs: {len(kzs):,}
- Therapy records: {len(therapy):,}
- SBP: {clean_primary['САД'].mean():.4f} mm Hg
- DBP: {clean_primary['ДАД'].mean():.4f} mm Hg
- Age: {clean_primary['возраст'].mean():.4f} years
""")
del future_meas
del invalid_age
del invalid_hr
import gc
gc.collect()

## STAGE 9: DETAILED EDA WITH ENHANCED VISUALIZATIONS

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime

# Load cleaned data with explicit types
clean_primary = pd.read_csv('data/primary_clean.csv', 
                           parse_dates=['время измерения', 'время сохранения на сервере', 'дата рождения пациента'])

# Create enhanced plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. SBP distribution with KDE
ax1 = axes[0, 0]
clean_primary['САД'].dropna().hist(bins=50, alpha=0.7, edgecolor='black', ax=ax1)
ax1.set_title('SBP Distribution\nMean: {:.1f}, Median: {:.1f}'.format(
    clean_primary['САД'].mean(), clean_primary['САД'].median()))
ax1.set_xlabel('SBP, mm Hg')
ax1.set_ylabel('Frequency')
ax1.axvline(clean_primary['САД'].mean(), color='red', linestyle='--', 
            label=f'Mean: {clean_primary["САД"].mean():.1f}')
ax1.axvline(clean_primary['САД'].median(), color='green', linestyle='--', 
            label=f'Median: {clean_primary["САД"].median():.1f}')
ax1.legend()

# 2. DBP distribution with KDE
ax2 = axes[0, 1]
clean_primary['ДАД'].dropna().hist(bins=50, alpha=0.7, edgecolor='black', color='green', ax=ax2)
ax2.set_title('DBP Distribution\nMean: {:.1f}, Median: {:.1f}'.format(
    clean_primary['ДАД'].mean(), clean_primary['ДАД'].median()))
ax2.set_xlabel('DBP, mm Hg')
ax2.set_ylabel('Frequency')
ax2.axvline(clean_primary['ДАД'].mean(), color='red', linestyle='--', 
            label=f'Mean: {clean_primary["ДАД"].mean():.1f}')
ax2.axvline(clean_primary['ДАД'].median(), color='green', linestyle='--', 
            label=f'Median: {clean_primary["ДАД"].median():.1f}')
ax2.legend()

# 3. SBP vs DBP relationship with correlation
ax3 = axes[1, 0]
sample = clean_primary.sample(min(10000, len(clean_primary)))
ax3.scatter(sample['САД'], sample['ДАД'], alpha=0.1, s=1)
corr = sample['САД'].corr(sample['ДАД'])
ax3.set_title(f'SBP vs DBP Relationship (sample 10k)\nPearson Correlation: {corr:.3f}')
ax3.set_xlabel('SBP, mm Hg')
ax3.set_ylabel('DBP, mm Hg')

# Add regression line
z = np.polyfit(sample['САД'].dropna(), sample['ДАД'].dropna(), 1)
p = np.poly1d(z)
ax3.plot(sample['САД'].sort_values(), p(sample['САД'].sort_values()), 
         "r--", alpha=0.8, label=f'Trend: y={z[0]:.2f}x+{z[1]:.1f}')
ax3.legend()

# 4. Distribution by hour (with data type check)
ax4 = axes[1, 1]

# Explicitly convert to datetime if not already
if not pd.api.types.is_datetime64_any_dtype(clean_primary['время измерения']):
    clean_primary['время измерения'] = pd.to_datetime(clean_primary['время измерения'], errors='coerce')

# Now extract hours
hours = clean_primary['время измерения'].dt.hour.dropna()
hour_counts = hours.value_counts().sort_index()

ax4.bar(hour_counts.index, hour_counts.values, alpha=0.7, color='orange', edgecolor='black')
ax4.set_title('Distribution of Measurements by Hour of Day')
ax4.set_xlabel('Hour of Day')
ax4.set_ylabel('Number of Measurements')
ax4.set_xticks(range(0, 24, 2))

# Add vertical lines for morning and evening peaks
ax4.axvline(x=9, color='red', linestyle='--', alpha=0.5, label='Morning (9:00)')
ax4.axvline(x=20, color='blue', linestyle='--', alpha=0.5, label='Evening (20:00)')
ax4.legend()

plt.tight_layout()
plt.savefig('data/photo/eda_detailed_plots.png', dpi=150)
plt.show()

print("\nPlot Analysis:")
print("="*50)

print("\n1. SBP Distribution:")
print(f"   - Mean: {clean_primary['САД'].mean():.1f} mm Hg")
print(f"   - Median: {clean_primary['САД'].median():.1f} mm Hg")
print(f"   - Standard deviation: {clean_primary['САД'].std():.1f}")
print(f"   - 25th percentile: {clean_primary['САД'].quantile(0.25):.1f}")
print(f"   - 75th percentile: {clean_primary['САД'].quantile(0.75):.1f}")

print("\n2. DBP Distribution:")
print(f"   - Mean: {clean_primary['ДАД'].mean():.1f} mm Hg")
print(f"   - Median: {clean_primary['ДАД'].median():.1f} mm Hg")
print(f"   - Standard deviation: {clean_primary['ДАД'].std():.1f}")
print(f"   - 25th percentile: {clean_primary['ДАД'].quantile(0.25):.1f}")
print(f"   - 75th percentile: {clean_primary['ДАД'].quantile(0.75):.1f}")

print(f"\n3. SBP and DBP Correlation: {corr:.3f}")
if corr > 0.7:
    print("   - Strong positive correlation (expected for BP)")
elif corr > 0.5:
    print("   - Moderate positive correlation")
else:
    print("   - Weak correlation (may require verification)")

# Analysis of measurement times
peak_morning = hour_counts[6:10].sum() if 6 in hour_counts.index else 0
peak_evening = hour_counts[18:22].sum() if 18 in hour_counts.index else 0
night_meas = sum(hour_counts[h] for h in range(0, 6) if h in hour_counts.index) + \
            sum(hour_counts[h] for h in range(22, 24) if h in hour_counts.index)

total_meas = len(hours)
print("\n4. Measurement Time Analysis:")
print(f"   - Morning hours (6-10): {peak_morning:,} measurements ({peak_morning/total_meas*100:.1f}%)")
print(f"   - Evening hours (18-22): {peak_evening:,} measurements ({peak_evening/total_meas*100:.1f}%)")
print(f"   - Night hours (22-6): {night_meas:,} measurements ({night_meas/total_meas*100:.1f}%)")
print(f"   - Daytime hours (10-18): {total_meas - peak_morning - peak_evening - night_meas:,} measurements ({(total_meas - peak_morning - peak_evening - night_meas)/total_meas*100:.1f}%)")

In [ ]:
plt.close(fig)  # Important! Close the figure
del sample
del hour_counts
del hours
del axes
import gc
gc.collect()

## STAGE 10: ANALYSIS BY OBSERVATION GROUPS

In [ ]:
# Make sure we have observation group
if 'группа наблюдения' not in clean_primary.columns:
    patient_program = pd.read_csv('data/patient_program.csv')
    clean_primary = clean_primary.merge(
        patient_program[['id пациента', 'группа наблюдения']].drop_duplicates('id пациента'),
        on='id пациента',
        how='left'
    )
    del patient_program  # delete after use

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
groups = clean_primary['группа наблюдения'].dropna().unique()
colors = {'experience': 'blue', 'control_1': 'green', 'undefined': 'orange'}

for i, group_name in enumerate(groups):
    group_data = clean_primary[clean_primary['группа наблюдения'] == group_name]
    
    if group_name == 'experience':
        label = 'Experience (active monitoring)'
    elif group_name == 'control_1':
        label = 'Control (without transmission)'
    else:
        label = 'Real practice'
    
    # SBP distribution by group
    axes[0, 0].hist(group_data['САД'].dropna(), bins=50, alpha=0.5, 
                    color=colors.get(group_name, 'gray'), label=label, density=True)
    
    # SBP Boxplot (without assignment)
    axes[0, 1].boxplot(group_data['САД'].dropna(), positions=[i], widths=0.6,
                       patch_artist=True, 
                       boxprops=dict(facecolor=colors.get(group_name, 'gray')))

axes[0, 0].set_title('SBP Distribution by Observation Group (normalized)')
axes[0, 0].set_xlabel('SBP, mm Hg')
axes[0, 0].set_ylabel('Density')
axes[0, 0].legend()

axes[0, 1].set_title('SBP Boxplot by Group')
axes[0, 1].set_xticks(range(len(groups)))
axes[0, 1].set_xticklabels(['Experience' if g=='experience' else 'Control' if g=='control_1' else 'Real practice' for g in groups])
axes[0, 1].set_ylabel('SBP, mm Hg')

# Number of measurements per patient by group
meas_per_patient = clean_primary.groupby(['группа наблюдения', 'id пациента']).size().reset_index(name='n_meas')
for group_name, group_data in meas_per_patient.groupby('группа наблюдения'):
    axes[1, 0].hist(group_data['n_meas'], bins=50, alpha=0.5, 
                    color=colors.get(group_name, 'gray'), 
                    label=('Experience' if group_name=='experience' else 'Control' if group_name=='control_1' else 'Real practice'), 
                    log=True)

axes[1, 0].set_title('Distribution of Measurements per Patient (log scale)')
axes[1, 0].set_xlabel('Number of Measurements')
axes[1, 0].set_ylabel('Frequency (log)')
axes[1, 0].legend()
axes[1, 0].set_xlim([0, 2000])

# Statistics by group
group_stats = []
for group_name, group_data in meas_per_patient.groupby('группа наблюдения'):
    group_stats.append({
        'Group': group_name,
        'Patients': len(group_data),
        'Mean measurements': f"{group_data['n_meas'].mean():.1f}",
        'Median measurements': f"{group_data['n_meas'].median():.1f}",
        'Min': group_data['n_meas'].min(),
        'Max': group_data['n_meas'].max(),
        'Total measurements': f"{group_data['n_meas'].sum():,}"
    })

group_stats_df = pd.DataFrame(group_stats)
print("\nMeasurement Statistics by Group:")
print(group_stats_df.to_string(index=False))

# Clear temporary statistics data
del meas_per_patient
del group_stats
del group_stats_df

# Correlation heatmap
numeric_cols = ['САД', 'ДАД', 'ЧП', 'возраст', 'рост', 'масса']
available_numeric = [col for col in numeric_cols if col in clean_primary.columns]
corr_matrix = clean_primary[available_numeric].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1, 1])
axes[1, 1].set_title('Correlation Matrix')

del corr_matrix  # clear correlation matrix
plt.tight_layout()
plt.savefig('data/photo/eda_group_analysis.png', dpi=150)
plt.show()
plt.close(fig)
del axes
import gc
gc.collect()

## STAGE 11: DATA QUALITY ANALYSIS

In [ ]:
# Missing values by group
print("\n1. Missing values in critical fields by group:")
missing_cols = ['рост', 'масса', 'основное заболевание', 'сопутствующие заболевание']
available_missing = [col for col in missing_cols if col in clean_primary.columns]

if available_missing:
    missing_by_group = clean_primary.groupby('группа наблюдения')[available_missing].agg(
        lambda x: x.isna().mean() * 100
    ).round(1)
    print(missing_by_group)
    del missing_by_group  # delete after output
else:
    print("   Missing fields not found in data")

# Measurement regularity
print("\n2. Measurement regularity:")
clean_primary['год_месяц'] = clean_primary['время измерения'].dt.to_period('M')
meas_by_month = clean_primary.groupby(['id пациента', 'год_месяц']).size().reset_index(name='n_meas_month')
patients_with_regular_meas = meas_by_month.groupby('id пациента').size()

print(f"   - Patients with measurements in 1 month: {(patients_with_regular_meas == 1).sum()}")
print(f"   - Patients with measurements in 2-6 months: {((patients_with_regular_meas >= 2) & (patients_with_regular_meas <= 6)).sum()}")
print(f"   - Patients with measurements in 7+ months: {(patients_with_regular_meas >= 7).sum()}")

# Clear temporary objects
del meas_by_month
del patients_with_regular_meas
clean_primary = clean_primary.drop('год_месяц', axis=1)

# Time delay
if 'время сохранения на сервере' in clean_primary.columns:
    clean_primary['задержка_сек'] = (pd.to_datetime(clean_primary['время сохранения на сервере']) - 
                                     pd.to_datetime(clean_primary['время измерения'])).dt.total_seconds()
    delay_stats = clean_primary['задержка_сек'].describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    print(f"\n3. Data transmission delay (seconds):")
    print(f"   - Median: {delay_stats['50%']:.0f} sec")
    print(f"   - 90th percentile: {delay_stats['90%']:.0f} sec")
    print(f"   - 95th percentile: {delay_stats['95%']:.0f} sec")
    print(f"   - 99th percentile: {delay_stats['99%']:.0f} sec")
    
    # Identify abnormally large delays
    large_delay = clean_primary[clean_primary['задержка_сек'] > 86400]  # > 1 day
    print(f"\n   - Measurements with delay > 1 day: {len(large_delay)} ({len(large_delay)/len(clean_primary)*100:.3f}%)")
    
    # Clear temporary objects
    clean_primary = clean_primary.drop('задержка_сек', axis=1)
    del large_delay
else:
    print("\n3. Delay information unavailable")

import gc
gc.collect()

## STAGE 12: DUPLICATE CLEANING AND NIGHT MEASUREMENT ANALYSIS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import gc

# Load data
clean_primary = pd.read_csv('data/primary_clean.csv', parse_dates=['время измерения', 'время сохранения на сервере'])
print(f"Measurements loaded: {len(clean_primary):,}")

# Data type check
print("\nData type check:")
print(clean_primary[['время измерения', 'время сохранения на сервере']].dtypes)

# Force conversion to datetime
clean_primary['время измерения'] = pd.to_datetime(clean_primary['время измерения'], errors='coerce')
clean_primary['время сохранения на сервере'] = pd.to_datetime(clean_primary['время сохранения на сервере'], errors='coerce')

# Remove rows with invalid dates
initial_len = len(clean_primary)
clean_primary = clean_primary.dropna(subset=['время измерения'])
print(f"Rows with invalid dates removed: {initial_len - len(clean_primary)}")

# 1. DUPLICATE REMOVAL
print("1. DUPLICATE REMOVAL")
duplicates_mask = clean_primary.duplicated(subset=['id пациента', 'время измерения'], keep=False)
duplicates = clean_primary[duplicates_mask].sort_values(['id пациента', 'время измерения'])
print(f"Duplicates found: {len(duplicates):,}")

if len(duplicates) > 0:
    print("\nDuplicate examples:")
    print(duplicates[['id пациента', 'время измерения', 'САД', 'ДАД', 'ЧП']].head(10))
    
    clean_primary = clean_primary.drop_duplicates(subset=['id пациента', 'время измерения'], keep='first')
    print(f"\nAfter duplicate removal: {len(clean_primary):,} measurements")
    print(f"Removed: {len(duplicates)//2:,} complete duplicates")
    
    del duplicates
    gc.collect()

# 2. NIGHT MEASUREMENT ANALYSIS
print("2. NIGHT MEASUREMENT ANALYSIS")
clean_primary['час_измерения'] = clean_primary['время измерения'].dt.hour
clean_primary['дата_измерения'] = clean_primary['время измерения'].dt.date

if 'часовой пояс пациента' in clean_primary.columns:
    print("\nTimezone information available")
    clean_primary['часовой пояс'] = pd.to_numeric(clean_primary['часовой пояс пациента'], errors='coerce')
    print("\nDistribution by timezone (all measurements):")
    print(clean_primary['часовой пояс'].value_counts().sort_index())
else:
    print("\nTimezone information not available")
    clean_primary['часовой пояс'] = 0

night_mask = (clean_primary['час_измерения'] >= 22) | (clean_primary['час_измерения'] < 6)

print(f"\nDaytime measurements: {len(clean_primary[~night_mask]):,} ({len(clean_primary[~night_mask])/len(clean_primary)*100:.1f}%)")
print(f"Night measurements: {len(clean_primary[night_mask]):,} ({len(clean_primary[night_mask])/len(clean_primary)*100:.1f}%)")

if len(clean_primary[night_mask]) > 0:
    timezone_night = clean_primary[night_mask].groupby('часовой пояс').size()
    print("\nAnalysis by timezone (night measurements):")
    print(timezone_night)
    
    print("\nPercentage of night measurements by timezone:")
    for tz in clean_primary['часовой пояс'].unique():
        tz_data = clean_primary[clean_primary['часовой пояс'] == tz]
        if len(tz_data) > 0:
            tz_night_pct = ((tz_data['час_измерения'] >= 22) | (tz_data['час_измерения'] < 6)).mean() * 100
            print(f"  Timezone {tz}: {tz_night_pct:.1f}% night measurements")
    
    night_per_patient = clean_primary[night_mask].groupby('id пациента').size().reset_index(name='night_measurements')
    total_per_patient = clean_primary.groupby('id пациента').size().reset_index(name='total_measurements')
    night_per_patient = night_per_patient.merge(total_per_patient, on='id пациента', how='right').fillna(0)
    night_per_patient['night_share'] = (night_per_patient['night_measurements'] / night_per_patient['total_measurements'] * 100).round(1)
    
    print("\nNight measurement statistics per patient:")
    print(f"  Mean night share: {night_per_patient['night_share'].mean():.1f}%")
    print(f"  Median night share: {night_per_patient['night_share'].median():.1f}%")
    print(f"  Patients with >50% night: {(night_per_patient['night_share'] > 50).sum()}")
    
    high_night_patients = night_per_patient[night_per_patient['night_share'] > 80]
    print(f"\nPatients with >80% night measurements: {len(high_night_patients)}")
    if len(high_night_patients) > 0:
        print("Examples of patients with abnormally high night measurement percentage:")
        print(high_night_patients.head(10))
    
    del night_per_patient, total_per_patient, high_night_patients
    gc.collect()

if 'время сохранения на сервере' in clean_primary.columns:
    clean_primary['задержка_сек'] = (clean_primary['время сохранения на сервере'] - clean_primary['время измерения']).dt.total_seconds()
    
    night_delay = clean_primary.loc[night_mask, 'задержка_сек'].median()
    day_delay = clean_primary.loc[~night_mask, 'задержка_сек'].median()
    
    print(f"\nMedian delay:")
    print(f"  Daytime measurements: {day_delay:.0f} sec")
    print(f"  Night measurements: {night_delay:.0f} sec")
    
    if night_delay > day_delay * 2:
        print("Night measurements have significantly higher delay")
else:
    print("\nDelay information not available")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

hour_counts = clean_primary['час_измерения'].value_counts().sort_index()
colors = ['orange' if (h < 6 or h >= 22) else 'skyblue' for h in hour_counts.index]
axes[0, 0].bar(hour_counts.index, hour_counts.values, color=colors, edgecolor='black')
axes[0, 0].set_title('Distribution of Measurements by Hour\n(orange - night hours)')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Number of Measurements')
axes[0, 0].set_xticks(range(0, 24, 2))
axes[0, 0].axvspan(22, 24, alpha=0.2, color='red', label='Night (22-24)')
axes[0, 0].axvspan(0, 6, alpha=0.2, color='red', label='Night (0-6)')
axes[0, 0].legend()

day_sad = clean_primary.loc[~night_mask, 'САД'].dropna()
night_sad = clean_primary.loc[night_mask, 'САД'].dropna()
axes[0, 1].boxplot([day_sad, night_sad], labels=['Day', 'Night'])
axes[0, 1].set_title('SBP: Day vs Night')
axes[0, 1].set_ylabel('SBP, mm Hg')

night_by_patient = clean_primary[night_mask].groupby('id пациента').size()
total_by_patient = clean_primary.groupby('id пациента').size()
night_share = (night_by_patient / total_by_patient * 100).fillna(0)
axes[1, 0].hist(night_share, bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Distribution of Night Measurement Share by Patient')
axes[1, 0].set_xlabel('Night Measurement Share (%)')
axes[1, 0].set_ylabel('Number of Patients')
axes[1, 0].axvline(x=50, color='red', linestyle='--', label='50%')
axes[1, 0].axvline(x=80, color='orange', linestyle='--', label='80%')
axes[1, 0].legend()

night_by_group = {}
for group in clean_primary['группа наблюдения'].unique():
    group_data = clean_primary[clean_primary['группа наблюдения'] == group]
    night_pct = ((group_data['час_измерения'] >= 22) | (group_data['час_измерения'] < 6)).mean() * 100
    night_by_group[group] = night_pct

groups_names = {'experience': 'Experience', 'control_1': 'Control', 'undefined': 'Real practice'}
x_labels = [groups_names.get(g, g) for g in night_by_group.keys()]
axes[1, 1].bar(x_labels, list(night_by_group.values()), color=['blue', 'green', 'orange'])
axes[1, 1].set_title('Night Measurement Share by Group')
axes[1, 1].set_ylabel('% Night Measurements')

plt.tight_layout()
plt.savefig('data/photo/night_measurements_analysis.png', dpi=150)
plt.show()
plt.close('all')

del hour_counts, day_sad, night_sad, night_share, night_by_group
gc.collect()

# 3. TIMEZONE CORRECTION
print("3. TIMEZONE CORRECTION")
if 'часовой пояс пациента' in clean_primary.columns:
    print("Timezone information available. Checking correction necessity...")
    
    clean_primary['часовой пояс'] = pd.to_numeric(clean_primary['часовой пояс пациента'], errors='coerce')
    
    print("\nDistribution by timezone:")
    print(clean_primary['часовой пояс'].value_counts().sort_index())
    
    clean_primary['время_локальное'] = clean_primary.apply(
        lambda row: row['время измерения'] + timedelta(hours=int(row['часовой пояс'] or 0)) 
        if pd.notna(row['часовой пояс']) else row['время измерения'], 
        axis=1
    )
    clean_primary['час_локальный'] = clean_primary['время_локальное'].dt.hour
    
    night_before = ((clean_primary['час_измерения'] >= 22) | (clean_primary['час_измерения'] < 6)).mean() * 100
    night_after = ((clean_primary['час_локальный'] >= 22) | (clean_primary['час_локальный'] < 6)).mean() * 100
    
    print(f"\nNight measurements before correction: {night_before:.1f}%")
    print(f"Night measurements after correction: {night_after:.1f}%")
    
    if night_after < night_before:
        print("Timezone correction reduced the share of night measurements")
        clean_primary['время измерения'] = clean_primary['время_локальное']
        clean_primary['час_измерения'] = clean_primary['час_локальный']
    else:
        print("Correction did not affect or worsened the situation - keeping original time")
    
    del clean_primary['время_локальное'], clean_primary['час_локальный']
else:
    print("Timezone information not available. Assuming time is already local.")
    print("Recommendation: for anomalous patients with >80% night measurements - check device settings")

# Save the result
clean_primary.to_csv('data/clean_primary_dedup.csv', index=False)
del clean_primary
gc.collect()

print("Analysis completed, memory cleared")

## STAGE 13: MISSING VALUE IMPUTATION

In [ ]:
# Load data
df = pd.read_csv('data/clean_primary_dedup.csv', parse_dates=['время измерения'])
print(f"Measurements loaded: {len(df):,}")
print(f"Unique patients: {df['id пациента'].nunique():,}")

# Create age groups
df['возраст_полных_лет'] = df['возраст'].round().astype('Int64')
bins = [0, 30, 40, 50, 60, 70, 80, 120]
labels = ['<30', '30-40', '40-50', '50-60', '60-70', '70-80', '80+']
df['возрастная_группа'] = pd.cut(df['возраст_полных_лет'], bins=bins, labels=labels)

print(f"\nMissing values before imputation:")
print(f"  Height: {df['рост'].isna().sum():,} ({df['рост'].isna().mean()*100:.1f}%)")
print(f"  Weight: {df['масса'].isna().sum():,} ({df['масса'].isna().mean()*100:.1f}%)")
print(f"  Primary disease: {df['основное заболевание'].isna().sum():,}")
print(f"  Comorbidities: {df['сопутствующие заболевание'].isna().sum():,}")

# Height and weight imputation (median by group + age)
df_imputed = df.copy()
for group in df_imputed['группа наблюдения'].unique():
    for age_group in labels:
        mask = (df_imputed['группа наблюдения'] == group) & (df_imputed['возрастная_группа'] == age_group)
        h_med = df_imputed.loc[mask, 'рост'].median()
        w_med = df_imputed.loc[mask, 'масса'].median()
        if pd.notna(h_med):
            df_imputed.loc[mask & df_imputed['рост'].isna(), 'рост'] = h_med
        if pd.notna(w_med):
            df_imputed.loc[mask & df_imputed['масса'].isna(), 'масса'] = w_med

# Global medians for remaining
df_imputed['рост'].fillna(df_imputed['рост'].median(), inplace=True)
df_imputed['масса'].fillna(df_imputed['масса'].median(), inplace=True)

# Disease imputation
df_imputed['основное заболевание'].fillna('Not specified', inplace=True)
df_imputed['сопутствующие заболевание'].fillna('No data', inplace=True)

# Create new features
df_imputed['ИМТ'] = (df_imputed['масса'] / ((df_imputed['рост']/100)**2)).round(1)
df_imputed['пульсовое_давление'] = df_imputed['САД'] - df_imputed['ДАД']

print(f"\nAfter imputation:")
print(f"  Missing values: {df_imputed.isnull().sum().sum()}")
print(f"  Mean BMI: {df_imputed['ИМТ'].mean():.1f}")
print(f"  Median BMI: {df_imputed['ИМТ'].median():.1f}")

df_imputed.to_csv('data/primary_imputed.csv', index=False)

# BMI CORRECTION AND VERIFICATION
# Limit to physiological boundaries
df_corrected = df_imputed.copy()
df_corrected.loc[df_corrected['масса'] > 150, 'масса'] = 150
df_corrected.loc[df_corrected['масса'] < 40, 'масса'] = 40
df_corrected.loc[df_corrected['рост'] > 210, 'рост'] = 210
df_corrected.loc[df_corrected['рост'] < 140, 'рост'] = 140
df_corrected['ИМТ'] = (df_corrected['масса'] / ((df_corrected['рост']/100)**2)).round(1)

print(f"\nBMI comparison before/after correction:")
print(f"  Before - mean: {df_imputed['ИМТ'].mean():.1f}, max: {df_imputed['ИМТ'].max():.1f}")
print(f"  After - mean: {df_corrected['ИМТ'].mean():.1f}, max: {df_corrected['ИМТ'].max():.1f}")

# Find anomalies
high_bmi = df_corrected[df_corrected['ИМТ'] > 50]['id пациента'].nunique()
print(f"\nPatients with BMI > 50: {high_bmi}")

# Distribution by category
bmi_bins = [0, 18.5, 25, 30, 35, 40, 100]
bmi_labels = ['Underweight', 'Normal', 'Overweight', 'Obesity 1', 'Obesity 2', 'Obesity 3']
df_corrected['ИМТ_категория'] = pd.cut(df_corrected['ИМТ'], bins=bmi_bins, labels=bmi_labels)
print("\nDistribution by BMI category:")
print(df_corrected['ИМТ_категория'].value_counts())

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Boxplot by group
df_corrected.boxplot(column='ИМТ', by='группа наблюдения', ax=axes[0,0])
axes[0,0].set_title('BMI by Group')

# Scatter height-weight
sample = df_corrected.sample(10000)
scatter = axes[0,1].scatter(sample['рост'], sample['масса'], c=sample['ИМТ'], cmap='viridis', alpha=0.5, s=1)
axes[0,1].set_xlabel('Height, cm')
axes[0,1].set_ylabel('Weight, kg')
axes[0,1].set_title('Height-Weight Relationship')
plt.colorbar(scatter, ax=axes[0,1])

# BMI by age
bmi_by_age = df_corrected.groupby('возрастная_группа')['ИМТ'].agg(['mean', 'median'])
bmi_by_age.plot(kind='bar', ax=axes[1,0])
axes[1,0].set_title('BMI by Age Groups')
axes[1,0].set_ylabel('BMI')

# BMI histogram
axes[1,1].hist(df_corrected['ИМТ'], bins=50, edgecolor='black', alpha=0.7)
axes[1,1].axvline(x=25, color='green', ls='--', label='Normal')
axes[1,1].axvline(x=30, color='orange', ls='--', label='Obesity')
axes[1,1].set_title('BMI Distribution')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('data/photo/bmi_analysis.png', dpi=150)
plt.show()
df_corrected.to_csv('data/primary_imputed_corrected.csv', index=False)
# Clear memory of intermediate data
del df_imputed 
del df  
del sample  
plt.tight_layout()
plt.savefig('data/photo/bmi_analysis.png', dpi=150)
plt.show()
plt.close() 
plt.close('all')  
import gc
gc.collect()

## STAGE 14: FINAL CORRECTIONS AND TARGET VARIABLE

In [ ]:
# Detailed analysis of group 30-40 years
group_30_40 = df_corrected[(df_corrected['возраст'] >= 30) & (df_corrected['возраст'] < 40)]
print("\nGroup 30-40 years by categories:")
for group in group_30_40['группа наблюдения'].unique():
    g_data = group_30_40[group_30_40['группа наблюдения'] == group]
    print(f"\n{group}:")
    print(f"  Patients: {g_data['id пациента'].nunique()}")
    print(f"  Mean weight: {g_data['масса'].mean():.1f} kg")
    print(f"  Mean BMI: {g_data['ИМТ'].mean():.1f}")

# Correlations by group
print("\nBMI-SBP correlation by group:")
for group in df_corrected['группа наблюдения'].unique():
    g_data = df_corrected[df_corrected['группа наблюдения'] == group]
    corr = g_data['ИМТ'].corr(g_data['САД'])
    print(f"  {group}: {corr:.3f}")

# Identify outliers
outliers = set()
outliers.update(df_corrected[df_corrected['ИМТ'] > 50]['id пациента'].unique())
outliers.update(df_corrected[(df_corrected['масса'] > 150) & (df_corrected['рост'] < 150)]['id пациента'].unique())
print(f"\nTotal outlier patients: {len(outliers)} ({len(outliers)/df_corrected['id пациента'].nunique()*100:.2f}%)")

# BMI class balance
print("\nBMI class balance:")
class_dist = df_corrected['ИМТ_категория'].value_counts()
for cat, count in class_dist.items():
    print(f"  {cat}: {count:,} ({count/len(df_corrected)*100:.1f}%)")
outliers.clear()
del outliers
gc.collect()

In [ ]:
# Exclude outliers
outlier_patients = set()
outlier_patients.update(df_corrected[df_corrected['ИМТ'] > 50]['id пациента'].unique())

few_measurements = df_corrected.groupby('id пациента').size()
outlier_patients.update(few_measurements[few_measurements < 5].index)

print(f"Patients excluded: {len(outlier_patients)}")
df_clean = df_corrected[~df_corrected['id пациента'].isin(outlier_patients)].copy()
print(f"Patients remaining: {df_clean['id пациента'].nunique()}")
print(f"Measurements: {len(df_clean):,}")
del df_corrected
del few_measurements

In [ ]:
# Feature aggregation
def iqr(x): return x.quantile(0.75) - x.quantile(0.25)
def days_diff(x): return (x.max() - x.min()).days

ml_features = df_clean.groupby('id пациента').agg({
    'возраст': 'first', 'рост': 'first', 'масса': 'first', 'ИМТ': 'first',
    'часовой пояс': 'first', 'группа наблюдения': 'first',
    'САД': ['mean', 'std', 'min', 'max', 'median', iqr],
    'ДАД': ['mean', 'std', 'min', 'max', 'median'],
    'ЧП': ['mean', 'std', 'min', 'max', 'median'],
    'пульсовое_давление': ['mean', 'std'],
    'время измерения': ['count', days_diff]
}).round(1)

ml_features.columns = [f"{col[0]}_{col[1]}" if col[1] not in ['first', 'count'] 
                       else f"{col[0]}" for col in ml_features.columns]
ml_features = ml_features.reset_index().rename(columns={'id пациента': 'patient_id'})
del df_clean

In [ ]:
# Add CSEs
kzs = pd.read_csv('data/kzs.csv')
kzs_per_patient = kzs.groupby('id пациента').size().reset_index(name='kzs_count')
ml_features = ml_features.merge(kzs_per_patient, left_on='patient_id', right_on='id пациента', how='left')
ml_features['kzs_count'].fillna(0, inplace=True)
ml_features.drop('id пациента', axis=1, inplace=True)
del kzs
del kzs_per_patient
print(f"\nFinal ML dataset: {ml_features.shape[0]} patients, {ml_features.shape[1]} features")
ml_features.to_csv('data/ml_dataset_final.csv', index=False)
del ml_features
gc.collect()

In [ ]:
ml_df = pd.read_csv('data/ml_dataset_final.csv')

# Fix missing ages
missing_age = ml_df['возраст'].isna().sum()
if missing_age > 0:
    median_age = ml_df.groupby('группа наблюдения')['возраст'].median()
    for group, med in median_age.items():
        ml_df.loc[(ml_df['группа наблюдения'] == group) & (ml_df['возраст'].isna()), 'возраст'] = med
    print(f"Missing ages filled: {missing_age}")
    del median_age  # delete after use

# Check experience group 30-40 years
high_bmi_exp = ml_df[(ml_df['группа наблюдения'] == 'experience') & 
                     (ml_df['ИМТ'] > 35) & 
                     (ml_df['возраст'].between(30, 40))]
print(f"\nExperience patients 30-40 years with BMI>35: {len(high_bmi_exp)}")
del high_bmi_exp  # delete temporary sample

# Final statistics (just printing, not creating variables)
print(f"\nFinal statistics:")
print(f"  Patients: {len(ml_df)}")
print(f"  Features: {len(ml_df.columns)}")
print(f"  BMI range: {ml_df['ИМТ'].min():.1f} - {ml_df['ИМТ'].max():.1f}")
print(f"  Age range: {ml_df['возраст'].min():.1f} - {ml_df['возраст'].max():.1f}")
print(f"  SBP range: {ml_df['САД_mean'].min():.1f} - {ml_df['САД_mean'].max():.1f}")

# Create target variables
median_kzs = ml_df['kzs_count'].median()
ml_df['target_high_risk'] = (ml_df['kzs_count'] > median_kzs).astype(int)
del median_kzs  # delete after use

bins = [0, 50, 100, 200, 1000]
labels = ['Low', 'Medium', 'High', 'Very High']
ml_df['target_risk_category'] = pd.cut(ml_df['kzs_count'], bins=bins, labels=labels)

print(f"\nTarget variables:")
print(f"  Binary: 0={sum(ml_df['target_high_risk']==0)}, 1={sum(ml_df['target_high_risk']==1)}")
print(f"  Multiclass:\n{ml_df['target_risk_category'].value_counts()}")
print(f"  Regression: {ml_df['kzs_count'].min()} - {ml_df['kzs_count'].max()}, mean={ml_df['kzs_count'].mean():.1f}")

# Save
ml_df.to_csv('data/ml_dataset_ready.csv', index=False)
del therapy
del ml_df
gc.collect()

In [ ]:
import os

files_to_delete = [
    'data/primary_clean.csv',
    'data/primary_imputed.csv',
    'data/primary_imputed_corrected.csv',
    'data/ml_dataset_final.csv',
    'data/therapy.csv'
]

for file in files_to_delete:
    if os.path.exists(file):
        os.remove(file)
        print(f"Deleted: {file}")
    else:
        print(f"Not found: {file}")